# PDF Text Extraction Pipeline

Pure PyMuPDF extraction with deterministic page routing and hash-based chunk deduplication.

**Pipeline:**
- `TEXT_PAGE` → plain text extraction
- `TABLE_PAGE` → block-level structured text (preserves column layout)
- `MATH_PAGE` → text + display equation tagging
- `VISUAL_PAGE` → caption extraction + figure metadata

**Output:** `data/chunks/<category>/<paper_id>.jsonl` — one JSON line per page chunk.

**Dedup:** SHA-256 hash index in `data/chunks/chunk_hashes.json` — skips exact duplicate pages across the corpus.

```bash
pip install pymupdf pdfplumber
```

## Cell 1 — Imports & Config

In [ ]:
import os
import re
import json
import hashlib
import logging
import time
from dataclasses import dataclass, asdict
from enum import Enum
from pathlib import Path
from typing import Optional

import fitz          # PyMuPDF
import pdfplumber    # table fallback

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
)
log = logging.getLogger(__name__)

# ── Repo root (notebook lives in empirical-rag-paradigm-benchmark/notebooks/) ──
REPO_ROOT      = Path("..")
PDF_ROOT       = REPO_ROOT / "data" / "arxiv" / "pdfs"   # contains cs.AI/, cs.DB/, …
CHUNKS_ROOT    = REPO_ROOT / "data" / "chunks"            # mirrors PDF_ROOT category structure
HASH_INDEX     = CHUNKS_ROOT / "chunk_hashes.json"        # global dedup index
ROUTING_LEDGER = CHUNKS_ROOT / "routing_ledger.jsonl"     # per-page routing log


MY_CATEGORIES = ["cs.AI", "cs.LG", "cs.IR", "cs.DB", "cs.SE"]

CHUNKS_ROOT.mkdir(parents=True, exist_ok=True)

# ── Router thresholds ──────────────────────────────────────────────────────
MIN_IMAGE_AREA_PX2      = 5_000   # px²; ignore decorative images below this
TABLE_BLOCK_COUNT       = 12      # many small blocks → likely table layout
TABLE_AVG_BLOCK_WIDTH_R = 0.40    # avg block width < 40% page width → columnar

# ── Regex ──────────────────────────────────────────────────────────────────
RE_TABLE_LABEL   = re.compile(r'\bTable\s+\d+', re.IGNORECASE)
RE_FIGURE_LABEL  = re.compile(r'\bFig(?:ure|\.)?\s+\d+', re.IGNORECASE)
# Display equations: indented line ending with a parenthesised number (equation label)
RE_DISPLAY_EQ    = re.compile(r'^\s{4,}.{1,120}\(\d+\)\s*$', re.MULTILINE)
# Figure caption lines: "Figure 3:" or "Fig. 3." at start of line
RE_CAPTION       = re.compile(r'^(?:Figure|Fig\.?)\s*\d+[.:]', re.IGNORECASE | re.MULTILINE)

print("Config loaded ✓")
print(f"PDF root  : {PDF_ROOT.resolve()}")
print(f"Chunks out: {CHUNKS_ROOT.resolve()}")

Config loaded ✓
PDF root  : C:\project\empirical-rag-paradigm-benchmark\data\arxiv\pdfs
Chunks out: C:\project\empirical-rag-paradigm-benchmark\data\chunks


## Cell 2 — Hash Index (Load / Save)

In [2]:
def load_hash_index() -> dict[str, str]:
    """
    Load the global chunk hash index.
    Maps  sha256_hex -> "paper_id:page_num"  (first occurrence wins).
    """
    if HASH_INDEX.exists():
        with HASH_INDEX.open() as f:
            return json.load(f)
    return {}


def save_hash_index(index: dict[str, str]) -> None:
    with HASH_INDEX.open("w") as f:
        json.dump(index, f)


def text_hash(text: str) -> str:
    """Full SHA-256 hex digest of UTF-8 encoded text."""
    return hashlib.sha256(text.encode("utf-8")).hexdigest()


# Load once at notebook start; passed into extract_pdf() calls
HASH_INDEX_DATA = load_hash_index()
print(f"Hash index loaded — {len(HASH_INDEX_DATA):,} known chunks")

Hash index loaded — 0 known chunks


## Cell 3 — Page Type Enum & Router

In [3]:
class PageType(str, Enum):
    TEXT   = "TEXT_PAGE"
    TABLE  = "TABLE_PAGE"
    MATH   = "MATH_PAGE"
    VISUAL = "VISUAL_PAGE"


def classify_page(page: fitz.Page) -> tuple[PageType, dict]:
    """
    Deterministic page router — explicit priority chain:
        VISUAL > TABLE > MATH > TEXT

    Returns (PageType, feature_dict) for the routing ledger.
    """
    page_width = page.rect.width

    # 1. Images (only count significant ones to ignore decorative elements)
    images = page.get_images(full=True)
    sig_images = [img for img in images if img[2] * img[3] >= MIN_IMAGE_AREA_PX2]

    # 2. Text blocks
    blocks      = page.get_text("blocks")
    text_blocks = [b for b in blocks if b[6] == 0]   # type 0 = text
    block_count = len(text_blocks)

    avg_bw_ratio = 0.0
    if block_count > 0 and page_width > 0:
        avg_bw_ratio = sum(b[2] - b[0] for b in text_blocks) / block_count / page_width

    # 3. Raw text signals
    raw_text        = page.get_text("text")
    has_table_label = bool(RE_TABLE_LABEL.search(raw_text))
    has_fig_label   = bool(RE_FIGURE_LABEL.search(raw_text))
    has_display_eq  = bool(RE_DISPLAY_EQ.search(raw_text))

    features = {
        "sig_image_count"    : len(sig_images),
        "block_count"        : block_count,
        "avg_bw_ratio"       : round(avg_bw_ratio, 4),
        "text_length"        : len(raw_text.strip()),
        "has_table_label"    : has_table_label,
        "has_fig_label"      : has_fig_label,
        "has_display_eq"     : has_display_eq,
    }

    # Priority chain ──────────────────────────────────────────────────────
    if len(sig_images) >= 1 or has_fig_label:
        return PageType.VISUAL, features

    if has_table_label or (
        block_count >= TABLE_BLOCK_COUNT
        and avg_bw_ratio < TABLE_AVG_BLOCK_WIDTH_R
    ):
        return PageType.TABLE, features

    if has_display_eq:
        return PageType.MATH, features

    return PageType.TEXT, features


print("Router defined ✓")

Router defined ✓


## Cell 4 — Page Extractors (pure PyMuPDF)

In [4]:
def extract_text_page(page: fitz.Page) -> str:
    """Standard text extraction — fast path for prose-heavy pages."""
    return page.get_text("text").strip()


def extract_table_page(page: fitz.Page) -> str:
    """
    Table pages: try pdfplumber for structured table extraction first.
    Fall back to PyMuPDF block-level text which preserves columnar layout
    better than the plain text mode.
    """
    # pdfplumber table attempt ─────────────────────────────────────────────
    try:
        import pdfplumber
        # pdfplumber needs to open the file; get it from the parent doc
        doc_path = page.parent.name
        if doc_path:  # empty string for in-memory docs
            with pdfplumber.open(doc_path) as pdf:
                pl_page  = pdf.pages[page.number]
                tables   = pl_page.extract_tables()
                if tables:
                    lines = []
                    for table in tables:
                        for row in table:
                            cells = [str(c or "").strip() for c in row]
                            lines.append(" | ".join(cells))
                        lines.append("")  # blank separator between tables
                    # Prepend any surrounding prose from PyMuPDF
                    prose = page.get_text("text").strip()
                    return (prose + "\n\n" + "\n".join(lines)).strip()
    except Exception:
        pass  # fall through to PyMuPDF block mode

    # PyMuPDF block-level fallback ─────────────────────────────────────────
    blocks = page.get_text("blocks")
    text_blocks = sorted(
        [b for b in blocks if b[6] == 0],
        key=lambda b: (round(b[1] / 20) * 20, b[0]),  # sort row-major
    )
    return "\n".join(b[4].strip() for b in text_blocks if b[4].strip())


def extract_math_page(page: fitz.Page) -> str:
    """
    Math pages: extract text and tag display equations explicitly so
    downstream chunking can identify them.
    """
    raw = page.get_text("text")
    # Tag numbered display equations: "   E = mc^2   (3)" → "[EQ] E = mc^2 (3)"
    tagged = RE_DISPLAY_EQ.sub(
        lambda m: "[EQ] " + m.group(0).strip(),
        raw,
    )
    return tagged.strip()


def extract_visual_page(page: fitz.Page) -> str:
    """
    Visual pages: extract surrounding text + caption lines.
    We cannot extract figure image content with PyMuPDF text mode,
    so we preserve captions and surrounding prose for RAG retrieval.
    """
    raw = page.get_text("text")
    # Tag caption lines for easier downstream parsing
    tagged = RE_CAPTION.sub(
        lambda m: "[CAPTION] " + m.group(0),
        raw,
    )
    # Also record embedded image metadata (dimensions, count)
    images = page.get_images(full=True)
    sig    = [img for img in images if img[2] * img[3] >= MIN_IMAGE_AREA_PX2]
    meta   = f"[FIGURES: {len(sig)} embedded image(s)]" if sig else ""
    return (meta + "\n" + tagged).strip() if meta else tagged.strip()


# Dispatch table — avoids if/elif chain in the hot loop
EXTRACTOR = {
    PageType.TEXT  : extract_text_page,
    PageType.TABLE : extract_table_page,
    PageType.MATH  : extract_math_page,
    PageType.VISUAL: extract_visual_page,
}

print("Extractors defined ✓")

Extractors defined ✓


## Cell 5 — Chunk Schema & Writers

In [5]:
@dataclass
class PageChunk:
    """
    One chunk = one page.
    Fields match the ingestion schema used by the PostgreSQL / MongoDB / Qdrant loaders.
    """
    paper_id     : str    # arXiv ID, e.g. "2301.07041"
    category     : str    # e.g. "cs.AI"
    page_num     : int    # 0-based
    page_type    : str    # PageType value
    text         : str    # extracted content
    char_count   : int
    content_hash : str    # full SHA-256 hex of text
    is_duplicate : bool   # True if this hash was seen earlier in the corpus
    source_path  : str    # relative path from repo root


def write_chunk(chunk: PageChunk, out_path: Path) -> None:
    """Append one chunk as a JSON line."""
    with out_path.open("a", encoding="utf-8") as f:
        f.write(json.dumps(asdict(chunk), ensure_ascii=False) + "\n")


def write_routing_entry(
    paper_id : str,
    category : str,
    page_num : int,
    page_type: PageType,
    features : dict,
) -> None:
    """Append one routing event to the global routing ledger."""
    entry = {
        "paper_id" : paper_id,
        "category" : category,
        "page_num" : page_num,
        "page_type": page_type.value,
        **features,
    }
    with ROUTING_LEDGER.open("a", encoding="utf-8") as f:
        f.write(json.dumps(entry) + "\n")


print("Schema & writers defined ✓")

Schema & writers defined ✓


## Cell 6 — Per-PDF Extraction

In [6]:
def extract_pdf(
    pdf_path     : Path,
    category     : str,
    hash_index   : dict[str, str],
    skip_existing: bool = True,
) -> dict:
    """
    Extract one PDF into a per-paper JSONL chunk file.

    Hash map logic:
      - compute SHA-256 of extracted page text
      - if hash exists in index → mark chunk as duplicate (still written,
        so the paper's JSONL is complete) but downstream ingestion can skip it
      - if new → register in index
    """
    paper_id = pdf_path.stem
    out_dir  = CHUNKS_ROOT / category
    out_dir.mkdir(parents=True, exist_ok=True)
    out_path = out_dir / f"{paper_id}.jsonl"

    if skip_existing and out_path.exists():
        log.debug("[SKIP] %s/%s", category, paper_id)
        return {"paper_id": paper_id, "category": category, "status": "skipped"}

    if out_path.exists():
        out_path.unlink()   # remove stale partial output before re-run

    counts    = {t: 0 for t in PageType}
    dup_count = 0
    t_start   = time.time()

    try:
        doc = fitz.open(str(pdf_path))
    except Exception as exc:
        log.error("[FAIL] %s/%s: %s", category, paper_id, exc)
        return {"paper_id": paper_id, "category": category, "status": "open_error", "error": str(exc)}

    for page_num, page in enumerate(doc):
        page_type, features = classify_page(page)
        counts[page_type] += 1

        # Extract using the appropriate strategy
        text = EXTRACTOR[page_type](page)

        # Hash-based dedup check
        h          = text_hash(text)
        first_seen = hash_index.get(h)
        is_dup     = first_seen is not None

        if not is_dup:
            hash_index[h] = f"{paper_id}:{page_num}"   # register first occurrence
        else:
            dup_count += 1

        chunk = PageChunk(
            paper_id     = paper_id,
            category     = category,
            page_num     = page_num,
            page_type    = page_type.value,
            text         = text,
            char_count   = len(text),
            content_hash = h,
            is_duplicate = is_dup,
            source_path  = str(pdf_path.relative_to(REPO_ROOT)),
        )
        write_chunk(chunk, out_path)
        write_routing_entry(paper_id, category, page_num, page_type, features)

    doc.close()
    elapsed = round(time.time() - t_start, 2)

    summary = {
        "paper_id" : paper_id,
        "category" : category,
        "status"   : "ok",
        "pages"    : sum(counts.values()),
        "duplicates": dup_count,
        "elapsed_s": elapsed,
        **{t.value: counts[t] for t in PageType},
    }
    log.info(
        "[OK] %s/%s | pages=%d dups=%d text=%d table=%d visual=%d math=%d (%.1fs)",
        category, paper_id,
        summary["pages"], dup_count,
        counts[PageType.TEXT], counts[PageType.TABLE],
        counts[PageType.VISUAL], counts[PageType.MATH],
        elapsed,
    )
    return summary


print("extract_pdf() defined ✓")

extract_pdf() defined ✓


## Cell 7 — Run the Pipeline

In [7]:
# ── Discover PDFs across my categories ────────────────────────────────────
pdf_queue: list[tuple[Path, str]] = []   # (pdf_path, category)
for cat in MY_CATEGORIES:
    cat_dir = PDF_ROOT / cat
    if not cat_dir.exists():
        log.warning("Category dir not found: %s", cat_dir)
        continue
    pdfs = sorted(cat_dir.glob("*.pdf"))
    pdf_queue.extend((p, cat) for p in pdfs)
    print(f"  {cat:<8} {len(pdfs):>4} PDFs")

print(f"\nTotal PDFs queued: {len(pdf_queue)}")

if not pdf_queue:
    print("\n⚠  No PDFs found. Check PDF_ROOT path.")
else:
    all_summaries = []
    total = len(pdf_queue)

    for i, (pdf_path, category) in enumerate(pdf_queue, 1):
        if i % 50 == 0 or i == 1:
            print(f"[{i}/{total}] {category}/{pdf_path.name}")

        summary = extract_pdf(
            pdf_path,
            category,
            hash_index    = HASH_INDEX_DATA,
            skip_existing = True,          # set False to force re-extract
        )
        all_summaries.append(summary)

    # ── Persist updated hash index ────────────────────────────────────────
    save_hash_index(HASH_INDEX_DATA)
    print(f"\nHash index saved — {len(HASH_INDEX_DATA):,} total unique chunks")

    # ── Summary report ────────────────────────────────────────────────────
    ok      = [s for s in all_summaries if s["status"] == "ok"]
    skipped = [s for s in all_summaries if s["status"] == "skipped"]
    failed  = [s for s in all_summaries if s["status"] not in ("ok", "skipped")]

    total_pages = sum(s.get("pages",       0) for s in ok)
    total_dups  = sum(s.get("duplicates",  0) for s in ok)
    total_time  = sum(s.get("elapsed_s",   0) for s in ok)

    print("\n" + "="*55)
    print("EXTRACTION SUMMARY")
    print("="*55)
    print(f"  PDFs processed : {len(ok)}")
    print(f"  Skipped        : {len(skipped)}")
    print(f"  Failed         : {len(failed)}")
    print(f"  Total pages    : {total_pages:,}")
    print(f"  Duplicate pages: {total_dups:,}  "
          f"({100*total_dups/max(total_pages,1):.1f}%)")
    print(f"  Elapsed        : {total_time:.1f}s")
    if failed:
        print("\n  Failed:")
        for s in failed:
            print(f"    {s['category']}/{s['paper_id']} — {s.get('error','?')}")
    print(f"\n  Chunks dir     : {CHUNKS_ROOT}")
    print(f"  Hash index     : {HASH_INDEX}")
    print(f"  Routing ledger : {ROUTING_LEDGER}")
    print("="*55)

  cs.AI     498 PDFs
  cs.LG     500 PDFs
  cs.IR     500 PDFs
  cs.DB     498 PDFs
  cs.SE     500 PDFs

Total PDFs queued: 2496
[1/2496] cs.AI/1709.02256.pdf


2026-06-18 18:30:18,251 [INFO] [OK] cs.AI/1709.02256 | pages=25 dups=0 text=17 table=8 visual=0 math=0 (0.7s)
2026-06-18 18:30:18,725 [INFO] [OK] cs.AI/1711.06301 | pages=13 dups=0 text=6 table=2 visual=5 math=0 (0.5s)
2026-06-18 18:30:19,545 [INFO] [OK] cs.AI/1803.08784 | pages=11 dups=0 text=0 table=6 visual=5 math=0 (0.8s)
2026-06-18 18:30:19,706 [INFO] [OK] cs.AI/1804.05044 | pages=15 dups=0 text=10 table=1 visual=4 math=0 (0.2s)
2026-06-18 18:30:19,987 [INFO] [OK] cs.AI/1902.01894 | pages=9 dups=0 text=1 table=1 visual=7 math=0 (0.3s)
2026-06-18 18:30:20,302 [INFO] [OK] cs.AI/1902.09980 | pages=33 dups=0 text=13 table=2 visual=18 math=0 (0.3s)
2026-06-18 18:30:20,743 [INFO] [OK] cs.AI/1906.06412 | pages=15 dups=0 text=8 table=3 visual=4 math=0 (0.4s)
2026-06-18 18:30:21,585 [INFO] [OK] cs.AI/1906.07900 | pages=14 dups=0 text=4 table=3 visual=7 math=0 (0.8s)
2026-06-18 18:30:21,735 [INFO] [OK] cs.AI/1907.04276 | pages=12 dups=0 text=3 table=0 visual=9 math=0 (0.1s)
2026-06-18 18:30

[50/2496] cs.AI/2012.15234.pdf


2026-06-18 18:31:00,564 [INFO] [OK] cs.AI/2012.15234 | pages=35 dups=0 text=16 table=1 visual=18 math=0 (0.3s)
2026-06-18 18:31:02,255 [INFO] [OK] cs.AI/2101.00058 | pages=40 dups=1 text=26 table=10 visual=4 math=0 (1.7s)
2026-06-18 18:31:03,272 [INFO] [OK] cs.AI/2101.06883 | pages=11 dups=0 text=2 table=5 visual=4 math=0 (1.0s)
2026-06-18 18:31:03,519 [INFO] [OK] cs.AI/2102.02959 | pages=20 dups=0 text=11 table=1 visual=8 math=0 (0.2s)
2026-06-18 18:31:04,493 [INFO] [OK] cs.AI/2102.03380 | pages=19 dups=0 text=15 table=4 visual=0 math=0 (1.0s)
2026-06-18 18:31:04,672 [INFO] [OK] cs.AI/2102.03406 | pages=24 dups=0 text=23 table=0 visual=1 math=0 (0.2s)
2026-06-18 18:31:05,706 [INFO] [OK] cs.AI/2102.03588 | pages=12 dups=0 text=3 table=4 visual=5 math=0 (1.0s)
2026-06-18 18:31:07,365 [INFO] [OK] cs.AI/2102.05147 | pages=59 dups=0 text=17 table=13 visual=29 math=0 (1.7s)
2026-06-18 18:31:09,432 [INFO] [OK] cs.AI/2102.06203 | pages=24 dups=0 text=3 table=11 visual=10 math=0 (2.1s)
2026-06

[100/2496] cs.AI/2106.06768.pdf


...

[2450/2496] cs.SE/2502.07950.pdf


2026-06-18 19:30:40,130 [INFO] [OK] cs.SE/2502.07956 | pages=5 dups=0 text=3 table=0 visual=2 math=0 (0.1s)
2026-06-18 19:30:40,670 [INFO] [OK] cs.SE/2502.07986 | pages=11 dups=0 text=4 table=1 visual=6 math=0 (0.5s)
2026-06-18 19:30:40,985 [INFO] [OK] cs.SE/2502.08050 | pages=8 dups=0 text=3 table=1 visual=4 math=0 (0.3s)
2026-06-18 19:30:43,947 [INFO] [OK] cs.SE/2502.08139 | pages=26 dups=0 text=11 table=11 visual=4 math=0 (3.0s)
2026-06-18 19:30:44,325 [INFO] [OK] cs.SE/2502.08172 | pages=13 dups=0 text=8 table=1 visual=4 math=0 (0.4s)
2026-06-18 19:30:45,218 [INFO] [OK] cs.SE/2502.08219 | pages=18 dups=0 text=8 table=3 visual=7 math=0 (0.9s)
2026-06-18 19:30:46,100 [INFO] [OK] cs.SE/2502.08224 | pages=10 dups=0 text=0 table=3 visual=7 math=0 (0.9s)
2026-06-18 19:30:49,128 [INFO] [OK] cs.SE/2502.08238 | pages=24 dups=0 text=12 table=11 visual=1 math=0 (3.0s)


MuPDF error: syntax error: could not parse color space (218 0 R)

MuPDF error: syntax error: could not parse color space (218 0 R)

MuPDF error: syntax error: could not parse color space (218 0 R)

MuPDF error: syntax error: could not parse color space (406 0 R)

MuPDF error: syntax error: could not parse color space (406 0 R)

MuPDF error: syntax error: could not parse color space (406 0 R)

MuPDF error: syntax error: could not parse color space (597 0 R)

MuPDF error: syntax error: could not parse color space (597 0 R)

MuPDF error: syntax error: could not parse color space (597 0 R)



2026-06-18 19:30:57,642 [INFO] [OK] cs.SE/2502.08341 | pages=15 dups=0 text=2 table=4 visual=9 math=0 (8.5s)
2026-06-18 19:30:59,174 [INFO] [OK] cs.SE/2502.08504 | pages=21 dups=0 text=14 table=5 visual=2 math=0 (1.5s)
2026-06-18 19:31:01,216 [INFO] [OK] cs.SE/2502.08802 | pages=7 dups=0 text=3 table=0 visual=4 math=0 (2.0s)
2026-06-18 19:31:02,461 [INFO] [OK] cs.SE/2502.08806 | pages=16 dups=0 text=1 table=5 visual=10 math=0 (1.2s)
2026-06-18 19:31:04,708 [INFO] [OK] cs.SE/2502.08925 | pages=62 dups=0 text=24 table=12 visual=26 math=0 (2.2s)
2026-06-18 19:31:04,989 [INFO] [OK] cs.SE/2502.09060 | pages=23 dups=0 text=15 table=0 visual=8 math=0 (0.3s)
2026-06-18 19:31:06,728 [INFO] [OK] cs.SE/2502.09146 | pages=34 dups=0 text=13 table=7 visual=14 math=0 (1.7s)
2026-06-18 19:31:08,015 [INFO] [OK] cs.SE/2502.09328 | pages=34 dups=0 text=11 table=6 visual=17 math=0 (1.3s)
2026-06-18 19:31:09,061 [INFO] [OK] cs.SE/2502.09771 | pages=13 dups=0 text=3 table=4 visual=6 math=0 (1.1s)
2026-06-18


Hash index saved — 43,134 total unique chunks

EXTRACTION SUMMARY
  PDFs processed : 2495
  Skipped        : 0
  Failed         : 1
  Total pages    : 43,211
  Duplicate pages: 77  (0.2%)
  Elapsed        : 3674.8s

  Failed:
    cs.LG/2501.01339 — Failed to open file '..\\data\\arxiv\\pdfs\\cs.LG\\2501.01339.pdf'.

  Chunks dir     : ..\data\chunks
  Hash index     : ..\data\chunks\chunk_hashes.json
  Routing ledger : ..\data\chunks\routing_ledger.jsonl


## Cell 8 — Inspect a Sample Chunk

In [8]:
# Pick the first available JSONL and print the first 5 chunks
sample_files = sorted(CHUNKS_ROOT.rglob("*.jsonl"))
sample_files = [f for f in sample_files if f.name != "routing_ledger.jsonl"]

if sample_files:
    sample_path = sample_files[0]
    print(f"Sample: {sample_path.relative_to(CHUNKS_ROOT)}\n")
    with sample_path.open() as f:
        for i, line in enumerate(f):
            c = json.loads(line)
            dup_flag = " [DUP]" if c["is_duplicate"] else ""
            print(f"--- p{c['page_num']} | {c['page_type']}{dup_flag} | chars={c['char_count']}")
            print(c["text"][:280].replace("\n", " ") + "…\n")
            if i >= 4:
                break
else:
    print("No chunk files yet — run Cell 7 first.")

Sample: cs.AI\1709.02256.jsonl



UnicodeDecodeError: 'charmap' codec can't decode byte 0x9d in position 2556: character maps to <undefined>

## Cell 9 — Routing Stats

In [9]:
from collections import Counter

if ROUTING_LEDGER.exists():
    type_counts = Counter()
    cat_counts  = Counter()
    total       = 0

    with ROUTING_LEDGER.open() as f:
        for line in f:
            e = json.loads(line)
            type_counts[e["page_type"]] += 1
            cat_counts[e["category"]]   += 1
            total += 1

    print(f"Total pages routed: {total:,}")
    print(f"Hash index size   : {len(HASH_INDEX_DATA):,} unique chunks\n")

    print("Page type breakdown:")
    for ptype, count in type_counts.most_common():
        bar = "█" * int(40 * count / total)
        print(f"  {ptype:<14} {count:>6}  {100*count/total:5.1f}%  {bar}")

    print("\nPages per category:")
    for cat, count in cat_counts.most_common():
        print(f"  {cat:<10} {count:>6}")
else:
    print("Routing ledger not found — run Cell 7 first.")

Total pages routed: 43,211
Hash index size   : 43,134 unique chunks

Page type breakdown:
  VISUAL_PAGE     17792   41.2%  ████████████████
  TEXT_PAGE       15748   36.4%  ██████████████
  TABLE_PAGE       9669   22.4%  ████████
  MATH_PAGE           2    0.0%  

Pages per category:
  cs.LG        9783
  cs.AI        9394
  cs.DB        9044
  cs.SE        8221
  cs.IR        6769
